Here, I verify the correctness of the values of the expectation value and matrix elements, taking into account that the Hamiltonian is being modified, and that the extended swap test is being used.

In [1]:
import numpy as np
import pickle
import scipy.sparse
from utils_ferm import (
    op_action_tz
)
from utils_CSF_and_UCSF import (
    orthogonal_transform_obt_tbt,
    obt_phys_spatial_to_spin,
    tbt_phys_spatial_to_spin,
    make_short_H_ferm_op
)
from utils_basic import shift_hamiltonian_qubits
from openfermion import (
    FermionOperator,
    jordan_wigner,
    QubitOperator,
    get_sparse_operator,
    normal_ordered
)

from utils_states import (
    somos_to_seniority_config,
    convert_TZ_format_to_sparse_format,
    variance_of_decomp,
    create_composite_state,
    create_composite_state_prepended,
    expectation,
    matrix_element,
    
)
from utils_hamiltonian_qubit import (
    process_qubit_hamiltonian_to_remove_irrelevant_terms,
    process_qubit_hamiltonian_to_project_onto_symmetric_subspace
)
from utils_hamiltonian_ferm import (
    process_fermionic_hamiltonian_to_remove_irrelevant_terms
)
from utils_partitioning import (
    sorted_insertion_decomposition,
    augment_decomp_with_pauli_x,
    augment_decomp_with_pauli_x_plus_i_pauli_y
)

filename = 'data/h2o_data.dump'

with open(filename,'rb') as f:
    (
    list_CSF,
    list_list_ia_CSF,
    list_list_theta_CSF,
    list_sym_CSF_vec,
    list_UCSF_tz,
    tz_states,
    somos_list,
    psi_GS_UCSF_smik,
    list_orb_rot,
    x_orbrot,
    Enuc,
    obt_spatial,
    tbt_spatial
    ) = pickle.load(f)

#Rotate orbitals 
if len(list_orb_rot) != 0:
    obt, tbt = orthogonal_transform_obt_tbt(x_orbrot,list_orb_rot,obt_spatial,tbt_spatial)
else:
    obt = obt_phys_spatial_to_spin(obt_spatial)
    tbt = tbt_phys_spatial_to_spin(tbt_spatial)


/Users/smikpatel/Documents/PhD/2025 - Summer/seniority/utils_states.py:28: SyntaxWarning: invalid escape sequence '\o'
  """
/Users/smikpatel/Documents/PhD/2025 - Summer/seniority/utils_states.py:57: SyntaxWarning: invalid escape sequence '\o'
  """


In [4]:
Nqubits = obt.shape[0]
dim     = 2 ** Nqubits
Norb    = Nqubits // 2
Nstates = len(tz_states)

# generate Hamiltonian and obtain its QWC decomposition, for both expectation values and matrix elements

Hferm = make_short_H_ferm_op(0, obt, tbt)
Hqub  = jordan_wigner(Hferm)
Hqub -= Hqub.constant
Hqub.compress()

qwc_decomp      = sorted_insertion_decomposition(Hqub, 'qwc')
qwc_decomp_swap = augment_decomp_with_pauli_x_plus_i_pauli_y(qwc_decomp, Nqubits)

# generate states and associated state information

configs      = [somos_to_seniority_config(somo, Norb) for somo in somos_list]
statevectors = [convert_TZ_format_to_sparse_format(dim, tz_state) for tz_state in tz_states]


for i in range(Nstates):
    for j in range(Nstates):
        if i >= j:

            # load the states 
            bra        = statevectors[i]
            bra_tz     = tz_states[i]
            bra_somos  = somos_list[i]
            bra_config = configs[i]

            ket        = statevectors[j]
            ket_tz     = tz_states[j]
            ket_somos  = somos_list[j]
            ket_config = configs[j]

            # process the Hamiltonian for measurements
            Hferm_processed = process_fermionic_hamiltonian_to_remove_irrelevant_terms(Hferm, Nqubits, bra, ket_tz)
            Hqub            = jordan_wigner(Hferm_processed)
            Hqub_processed  = process_qubit_hamiltonian_to_remove_irrelevant_terms(Hqub, Nqubits, bra_config, ket_config)
            Hqub_tapered    = process_qubit_hamiltonian_to_project_onto_symmetric_subspace(Hqub, Nqubits, bra_config, ket_config)

            # obtain Hamiltonian fragments
            qwc_decomp_processed      = sorted_insertion_decomposition(Hqub_processed, 'qwc')
            qwc_decomp_processed_swap = augment_decomp_with_pauli_x_plus_i_pauli_y(qwc_decomp_processed, Nqubits)

            # do a bunch of correctness checks

            # first: should get the same matrix element no matter how Hamiltonian is processed
            matrix_element1 = np.round((bra @ get_sparse_operator(Hferm, Nqubits)           @ ket.T)[0,0], 8)
            matrix_element2 = np.round((bra @ get_sparse_operator(Hferm_processed, Nqubits) @ ket.T)[0,0], 8)
            matrix_element3 = np.round((bra @ get_sparse_operator(Hqub, Nqubits)            @ ket.T)[0,0], 8)
            matrix_element4 = np.round((bra @ get_sparse_operator(Hqub_processed, Nqubits)  @ ket.T)[0,0], 8)

            assert matrix_element1 == matrix_element2
            assert matrix_element1 == matrix_element3
            assert matrix_element1 == matrix_element4
            assert matrix_element2 == matrix_element3
            assert matrix_element2 == matrix_element4
            assert matrix_element3 == matrix_element4
                
            # second: should get the same matrix element from the fragments, including with swap test
            H_from_fragments           = sum(qwc_decomp_processed)
            H_tensor_1q_from_fragments = sum(qwc_decomp_processed_swap)
            bra_ket_composite          = create_composite_state(bra, ket, Nqubits)

            assert (Hqub_processed - H_from_fragments) == QubitOperator().zero()
            assert (Hqub_processed * (QubitOperator(f'X{Nqubits}') + 1j * QubitOperator(f'Y{Nqubits}')) - H_tensor_1q_from_fragments) == QubitOperator().zero()


            matrix_element5 = np.round((bra @ get_sparse_operator(H_from_fragments, Nqubits) @ ket.T)[0,0], 8)
            matrix_element6 = np.round((bra_ket_composite @ get_sparse_operator(H_tensor_1q_from_fragments, Nqubits + 1) @ bra_ket_composite.T)[0,0], 8)

            assert matrix_element1 == matrix_element5
            assert matrix_element1 == matrix_element6
            assert matrix_element2 == matrix_element5
            assert matrix_element2 == matrix_element6
            assert matrix_element3 == matrix_element5
            assert matrix_element3 == matrix_element6
            assert matrix_element4 == matrix_element5
            assert matrix_element4 == matrix_element6
            assert matrix_element5 == matrix_element6

KeyboardInterrupt: 